In [10]:
import torch as t
from abc import ABC, abstractmethod
from typing import List
import einops
import sys
import os
import numpy as np
import random
from pathlib import Path

# Add the parent directory to path so we can import modular_addition
parent_dir = os.path.abspath('..')
sys.path.insert(0, parent_dir)

# Import MLP directly and make it available in the modular_addition namespace
from modular_addition.model import MLP
import modular_addition
modular_addition.MLP = MLP  # Make MLP accessible via modular_addition

modular_addition_path = os.path.abspath('../modular_addition')
sys.path.insert(0, modular_addition_path)

# Now these imports should work
from model_mlp_ihvp import MLP_IHVP, ExperimentParams
from dataset import make_dataset, train_test_split, make_random_dataset
from compute_fourier_peakness import find_recursive_ntk_neighbors, compute_empirical_ntk, analyze_jacobian_fourier_peakiness, MLP, compute_eigenvalues
from influence_functions_mlp import get_query_grad, get_influences, dataset_sample, get_ekfac_factors_and_pseudo_grads, get_grads, get_ekfac_ihvp

seed_int = 422
t.manual_seed(seed_int)
np.random.seed(seed_int)
random.seed(seed_int)

In [11]:
params = ExperimentParams(
    linear_1_tied=True,
    tie_unembed=False,
    movie=True,
    scale_linear_1_factor=1.0,
    scale_embed=1.0,
    use_random_dataset=False,
    freeze_middle=False,
    n_batches=2000,
    n_save_model_checkpoints=40,
    lr=0.005,
    magnitude=False,
    ablation_fourier=False,
    do_viz_weights_modes=False,
    batch_size=128,
    num_no_weight_decay_steps=0,
    weight_decay=0,
    run_id=0,
    activation="quad",
    hidden_size=32,
    embed_dim=16,
    train_frac=0.95
)
model = MLP(params)

checkpoint_path = Path(modular_addition_path) / "models/checkpoints/CHECKPOINT_39_P53_frac0.95_hid32_emb16_tieunembedFalse_tielinTrue_freezeFalse_run0.pt"

# Load the model state
model.load_state_dict(t.load(checkpoint_path))
model.to(params.device)

model_ihvp = MLP_IHVP(params)

checkpoint_path = Path(modular_addition_path) / "models_mlp/checkpoints/CHECKPOINT_39_P53_frac0.95_hid32_emb16_tieunembedFalse_tielinTrue_freezeFalse_run0.pt"

# Load the model state
model_ihvp.load_state_dict(t.load(checkpoint_path))
model_ihvp.to(params.device)

print(model)
print(model_ihvp)

MLP(
  (embedding): Embedding(53, 16)
  (linear1r): Linear(in_features=16, out_features=32, bias=True)
  (linear1l): Linear(in_features=16, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=53, bias=False)
)
MLP_IHVP(
  (embedding): Embedding(53, 16)
  (fc1): MLPBlock(
    (linear): Linear(in_features=16, out_features=32, bias=True)
  )
  (fc2): MLPBlock(
    (linear): Linear(in_features=32, out_features=53, bias=False)
  )
)


In [12]:
if params.use_random_dataset:
    dataset = make_random_dataset(params.p, params.random_seed)
else:
    dataset = make_dataset(params.p)
train_data, test_data = train_test_split(
    dataset, params.train_frac, params.random_seed
)

print("num train data", len(train_data))
print("num test data", len(test_data))

First indices of shuffled dataset [2427, 2011, 1269, 1582, 2369, 1755, 2468, 186, 1077, 2448]
num train data 2668
num test data 141


In [20]:
torch_matrix, nd_array, jacobian = compute_empirical_ntk(model, t.nn.CrossEntropyLoss(), train_data)
# ntk matrix is symmetric, so we can use the sum of the matrix and its transpose for eigenvalues for obtainingreal eigenvalues
sorted_eigenvals, effective_dim = compute_eigenvalues(nd_array + nd_array.T)
print("Sorted_eigenvals of the NTK matrix + its transpose", sorted_eigenvals[:50])
print("Effective dimension of the NTK matrix + its transpose", effective_dim, "out of", nd_array.shape[0])


Sorted_eigenvals of the NTK matrix + its transpose [4.1313214  3.3120658  1.7049577  1.4998667  1.2581177  1.1992072
 1.0716637  1.058979   0.8510709  0.83749574 0.80681545 0.7634225
 0.7487769  0.73190594 0.66803473 0.6609711  0.6461573  0.6344052
 0.61326534 0.5744884  0.5705872  0.55992645 0.5372406  0.5009574
 0.49244478 0.47016257 0.45662335 0.43541372 0.3956958  0.3880659
 0.37725785 0.35647678 0.35484555 0.35098222 0.34737757 0.33792758
 0.3328704  0.32380638 0.31770283 0.30655658 0.29086554 0.28452715
 0.27641568 0.26976115 0.26342288 0.25958225 0.24886894 0.24452059
 0.23814964 0.23407507]
Effective dimension of the NTK matrix + its transpose 127.80892 out of 2668


In [14]:
sampled_indices = np.random.choice(torch_matrix.shape[0], 5, replace=False).tolist()
num_neighbors = 20

list_of_ntk_circuit_indices = []
for start_idx in sampled_indices:
    circuit_indices = find_recursive_ntk_neighbors(
        torch_matrix, 
        start_idx,
        neighbors_per_step=3,
        total_neighbors=num_neighbors,
        strategy="last"
    )
    list_of_ntk_circuit_indices.append(circuit_indices)

print("list_of_ntk_circuit_indices", list_of_ntk_circuit_indices)

list_of_ntk_circuit_indices [[1770, 1821, 2444, 751, 1955, 1399, 337, 2171, 809, 69, 2106, 965, 2640, 2667, 1923, 1661, 1831, 2533, 1766, 66], [1070, 835, 368, 821, 1265, 220, 2535, 2559, 1491, 590, 1805, 438, 411, 2001, 511, 1597, 571, 2506, 1586, 790], [2609, 2489, 2176, 1374, 1865, 1705, 429, 1219, 2565, 597, 2448, 541, 1255, 2333, 1237, 128, 1112, 2071, 1587, 511], [423, 757, 2057, 458, 1181, 2318, 273, 492, 2078, 1513, 1989, 2145, 1688, 1236, 652, 637, 2001, 511, 2436, 571], [2640, 2667, 1831, 1661, 1923, 1572, 66, 943, 752, 1766, 2533, 2449, 825, 2163, 1474, 1384, 1924, 1565, 1187, 238]]


In [15]:
for circuit_indices in list_of_ntk_circuit_indices:

    # Analyze the Jacobian for this circuit
    jacobian_circuit = jacobian[circuit_indices]
    U, S, Vh = t.linalg.svd(jacobian_circuit)
    
    # Find the maximum singular value
    max_singular_value = S[0]
    
    # Count values less than 10% of the maximum
    threshold = 0.1 * max_singular_value
    small_values_count = (S < threshold).sum().item()
    small_values_percentage = (small_values_count / len(S)) * 100
    
    print(f"Max singular value: {max_singular_value:.4f}")
    print(f"Number of values < 10% of max: {small_values_count}/{len(S)} ({small_values_percentage:.1f}%)")
    
    # Optional: Analyze the top weight directions (right singular vectors)
    top_weight_directions = Vh[:3]  # Top 3 directions in weight space
    
    # TODO: You could further analyze these directions, e.g.:
    # 1. Identify which weights have the largest components
    # 2. Check if these align with specific Fourier modes
    # 3. Visualize the pattern of weights being used
    
    print("-" * 50)
    
    # Analyze Fourier peakiness of the circuit's gradients
    fourier_results, overall_results = analyze_jacobian_fourier_peakiness(jacobian_circuit, model, top_k_modes=5, module_name="embedding")
    
    # Print the most peaked parameters
    print(f"Fourier analysis for circuit starting at {start_idx}:")
    
    # Sort results by peakiness
    sorted_results = sorted(
        [(name, data) for name, data in fourier_results.items()],
        key=lambda x: x[1]['peakiness'],
        reverse=True
    )
    
    # Print top 5 most peaked parameters
    for name, data in sorted_results[:5]:
        print(f"  {name}:")
        print(f"    Top modes: {data['top_modes']}")
        print(f"    Peakiness: {data['peakiness']:.2f}")
        print(f"    Concentration: {data['concentration']:.2%}")
        
    print(f" top modes overall: {overall_results['overall']['top_modes']}")
    print(f" peakiness overall: {overall_results['overall']['peakiness']}")
    print(f" concentration overall: {overall_results['overall']['concentration']}")
    # Analyze Fourier peakiness of the circuit's gradients
    fourier_results, overall_results = analyze_jacobian_fourier_peakiness(jacobian_circuit, model, top_k_modes=5, module_name="linear1")
    
    # Print the most peaked parameters
    print(f"Fourier analysis for circuit starting at {start_idx}:")
    
    # Sort results by peakiness
    sorted_results = sorted(
        [(name, data) for name, data in fourier_results.items()],
        key=lambda x: x[1]['peakiness'],
        reverse=True
    )
    
    # Print top 5 most peaked parameters
    for name, data in sorted_results[:5]:
        print(f"  {name}:")
        print(f"    Top modes: {data['top_modes']}")
        print(f"    Peakiness: {data['peakiness']:.2f}")
        print(f"    Concentration: {data['concentration']:.2%}")
        
    print(f" top modes overall: {overall_results['overall']['top_modes']}")
    print(f" peakiness overall: {overall_results['overall']['peakiness']}")
    print(f" concentration overall: {overall_results['overall']['concentration']}")

    print("-" * 50)

Max singular value: 0.4444
Number of values < 10% of max: 8/20 (40.0%)
--------------------------------------------------
Fourier analysis for circuit starting at 2640:
  embedding.weight[19]:
    Top modes: [(8, 0.20973646640777588), (11, 0.1106766015291214), (5, 0.1106766015291214), (14, 0.08679650723934174), (2, 0.08679650723934174)]
    Peakiness: 3.98
    Concentration: 20.97%
  embedding.weight[30]:
    Top modes: [(8, 0.20973646640777588), (11, 0.1106766015291214), (5, 0.1106766015291214), (14, 0.08679650723934174), (2, 0.08679650723934174)]
    Peakiness: 3.98
    Concentration: 20.97%
  embedding.weight[21]:
    Top modes: [(8, 0.13007564842700958), (14, 0.12046005576848984), (2, 0.12046005576848984), (10, 0.08871650695800781), (6, 0.08871650695800781)]
    Peakiness: 2.24
    Concentration: 13.01%
  embedding.weight[28]:
    Top modes: [(8, 0.13007564842700958), (14, 0.12046005576848984), (2, 0.12046005576848984), (10, 0.08871650695800781), (6, 0.08871650695800781)]
    Peaki

In [16]:
model_ihvp.eval()
n_queries = 10
queries = dataset_sample(test_data, n_queries)
gradient_fitting_data = dataset_sample(train_data, len(train_data))
search_data = dataset_sample(train_data, len(train_data))

mlp_blocks = [model_ihvp.fc1, model_ihvp.fc2]


topk = 10
print("len of queries", len(queries))
print("len of gradient fitting data", len(gradient_fitting_data))
print("len of search data", len(search_data))
print("first 5 queries", queries[0:5])
print("first 5 gradient fitting data", gradient_fitting_data[0:5])
print("first 5 search data", search_data[0:5])


len of queries 10
len of gradient fitting data 2668
len of search data 2668
first 5 queries [((tensor(26), tensor(28)), tensor(1)), ((tensor(6), tensor(30)), tensor(36)), ((tensor(27), tensor(8)), tensor(35)), ((tensor(23), tensor(34)), tensor(4)), ((tensor(18), tensor(49)), tensor(14))]
first 5 gradient fitting data [((tensor(41), tensor(41)), tensor(29)), ((tensor(48), tensor(37)), tensor(32)), ((tensor(6), tensor(43)), tensor(49)), ((tensor(34), tensor(26)), tensor(7)), ((tensor(8), tensor(44)), tensor(52))]
first 5 search data [((tensor(35), tensor(6)), tensor(41)), ((tensor(41), tensor(15)), tensor(3)), ((tensor(43), tensor(41)), tensor(31)), ((tensor(37), tensor(21)), tensor(5)), ((tensor(33), tensor(45)), tensor(25))]


In [17]:
kfac_input_covs, kfac_grad_covs, pseudo_grads = get_ekfac_factors_and_pseudo_grads(
    model_ihvp, gradient_fitting_data, mlp_blocks, params.device
)

search_grads = get_grads(model_ihvp, search_data, mlp_blocks, params.device)

ihvp = get_ekfac_ihvp(kfac_input_covs, kfac_grad_covs, pseudo_grads, search_grads)

print("ihvp.shape", ihvp.shape)

torch_matrix, nd_array, jacobian = compute_empirical_ntk(model_ihvp, t.nn.CrossEntropyLoss(), train_data)

print("torch_matrix.shape", torch_matrix.shape)
print("nd_array.shape", nd_array.shape)
print("jacobian.shape", jacobian.shape)

sorted_eigenvals, effective_dim = compute_eigenvalues(torch_matrix + torch_matrix.T)
print("Sorted_eigenvals of the NTK matrix + its transpose", sorted_eigenvals[:50])
print("Effective dimension of the NTK matrix + its transpose", effective_dim, "out of", nd_array.shape[0])


ihvp.shape torch.Size([2668, 2293])
torch_matrix.shape torch.Size([2668, 2668])
nd_array.shape (2668, 2668)
jacobian.shape torch.Size([2668, 3088])
Sorted_eigenvals of the NTK matrix + its transpose [1.2483044  1.2163085  1.1174465  1.0236914  0.8789867  0.82496256
 0.78570753 0.6636851  0.583899   0.52588964 0.5073956  0.48359117
 0.47839722 0.4681793  0.42548227 0.39098427 0.38650465 0.3851478
 0.33735067 0.3249536  0.31043786 0.2976887  0.2939055  0.29094774
 0.28368852 0.2730129  0.2718682  0.25820324 0.2488611  0.24547522
 0.24165152 0.23814702 0.2266972  0.22163434 0.1969594  0.19469911
 0.1879921  0.18450573 0.17863461 0.17623557 0.17329717 0.16715646
 0.1624731  0.15959918 0.15826838 0.154044   0.15235715 0.14946255
 0.14239874 0.14029741]
Effective dimension of the NTK matrix + its transpose 131.43788 out of 2668


In [18]:
U, S, Vh = t.linalg.svd(ihvp)
# Find the maximum singular value
max_singular_value = S[0]
print("top 50 singular values for ihvp", S[:50])
# Count values less than 10% of the maximum
threshold = 0.1 * max_singular_value
small_values_count = (S < threshold).sum().item()
small_values_percentage = (small_values_count / len(S)) * 100

print(f"Max singular value for ihvp: {max_singular_value:.4f}")
print(f"Number of values < 10% of max for ihvp: {small_values_count}/{len(S)} ({small_values_percentage:.1f}%)")

# Normalize eigenvalues to create a probability distribution
S_numpy = S.detach().cpu().numpy()
total = np.sum(S_numpy)
normalized_singular_values = S_numpy / total

# compute entropy and effective dimension
nonzero_probs = normalized_singular_values[normalized_singular_values > 0]
entropy = -np.sum(nonzero_probs * np.log(nonzero_probs))
effective_dim = np.exp(entropy)

print(f"Effective dimension for ihvp: {effective_dim:.2f} (out of {len(S)})")

U, S, Vh = t.linalg.svd(jacobian)
# Find the maximum singular value
max_singular_value = S[0]
print("top 50 singular values for jacobian", S[:50])
# Count values less than 10% of the maximum
threshold = 0.1 * max_singular_value
small_values_count = (S < threshold).sum().item()
small_values_percentage = (small_values_count / len(S)) * 100

print(f"Max singular value for jacobian: {max_singular_value:.4f}")
print(f"Number of values < 10% of max for jacobian: {small_values_count}/{len(S)} ({small_values_percentage:.1f}%)")

# Normalize eigenvalues to create a probability distribution
S_numpy = S.detach().cpu().numpy()
total = np.sum(S_numpy)
normalized_singular_values = S_numpy / total
# compute entropy and effective dimension
nonzero_probs = normalized_singular_values[normalized_singular_values > 0]
entropy = -np.sum(nonzero_probs * np.log(nonzero_probs))
effective_dim = np.exp(entropy)

print(f"Effective dimension for jacobian: {effective_dim:.2f} (out of {len(S)})")


top 50 singular values for ihvp tensor([0.0008, 0.0008, 0.0007, 0.0007, 0.0007, 0.0006, 0.0006, 0.0006, 0.0005,
        0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0004, 0.0004, 0.0004,
        0.0004, 0.0004, 0.0004, 0.0004, 0.0004, 0.0004, 0.0004, 0.0004, 0.0004,
        0.0004, 0.0004, 0.0003, 0.0003, 0.0003, 0.0003, 0.0003, 0.0003, 0.0003,
        0.0003, 0.0003, 0.0003, 0.0003, 0.0003, 0.0003, 0.0003, 0.0003, 0.0003,
        0.0003, 0.0003, 0.0003, 0.0003, 0.0003], device='cuda:0')
Max singular value for ihvp: 0.0008
Number of values < 10% of max for ihvp: 2072/2293 (90.4%)
Effective dimension for ihvp: 399.87 (out of 2293)
top 50 singular values for jacobian tensor([0.7900, 0.7798, 0.7475, 0.7154, 0.6630, 0.6423, 0.6269, 0.5761, 0.5404,
        0.5128, 0.5037, 0.4917, 0.4891, 0.4838, 0.4612, 0.4421, 0.4395, 0.4387,
        0.4107, 0.4030, 0.3939, 0.3858, 0.3833, 0.3814, 0.3766, 0.3694, 0.3687,
        0.3593, 0.3528, 0.3503, 0.3476, 0.3451, 0.3366, 0.3329, 0.3138, 0.3120,
 

In [19]:
query_grads = []
for i, data in enumerate(search_data):
    query_grad = get_query_grad(model_ihvp, data, mlp_blocks, params.device)
    query_grads.append(query_grad)
query_grads = t.stack(query_grads)
influences_matrix = ihvp @ query_grads.T

U, S, Vh = t.linalg.svd(influences_matrix)
# Find the maximum singular value
max_singular_value = S[0]
print("singular values for influences_matrix", S[:50])
# Count values less than 10% of the maximum
threshold = 0.1 * max_singular_value
small_values_count = (S < threshold).sum().item()
small_values_percentage = (small_values_count / len(S)) * 100

print(f"Max singular value for influences_matrix: {max_singular_value:.4f}")
print(f"Number of values < 10% of max for influences_matrix: {small_values_count}/{len(S)} ({small_values_percentage:.1f}%)")

# Normalize eigenvalues to create a probability distribution
S_numpy = S.detach().cpu().numpy()
total = np.sum(S_numpy)
normalized_singular_values = S_numpy / total
# compute entropy and effective dimension
nonzero_probs = normalized_singular_values[normalized_singular_values > 0]
entropy = -np.sum(nonzero_probs * np.log(nonzero_probs))

effective_dim = np.exp(entropy)

print(f"Effective dimension for influences_matrix: {effective_dim:.2f} (out of {len(S)})")

singular values for influences_matrix tensor([9.8714e-05, 7.9832e-05, 7.7950e-05, 6.6701e-05, 6.4828e-05, 5.9744e-05,
        5.4960e-05, 5.3324e-05, 5.1847e-05, 4.9372e-05, 4.7978e-05, 4.2837e-05,
        4.1327e-05, 3.9835e-05, 3.8741e-05, 3.8080e-05, 3.7364e-05, 3.5187e-05,
        3.4786e-05, 3.3514e-05, 3.1567e-05, 3.1060e-05, 3.0671e-05, 2.9612e-05,
        2.8250e-05, 2.8171e-05, 2.7583e-05, 2.6611e-05, 2.6021e-05, 2.5696e-05,
        2.5284e-05, 2.4509e-05, 2.3417e-05, 2.3367e-05, 2.2799e-05, 2.2439e-05,
        2.2113e-05, 2.1395e-05, 2.0975e-05, 2.0742e-05, 1.9639e-05, 1.9433e-05,
        1.9357e-05, 1.9157e-05, 1.8893e-05, 1.8439e-05, 1.7931e-05, 1.7707e-05,
        1.7411e-05, 1.7216e-05], device='cuda:0')
Max singular value for influences_matrix: 0.0001
Number of values < 10% of max for influences_matrix: 2584/2668 (96.9%)
Effective dimension for influences_matrix: 200.51 (out of 2668)
